# Week 3 - Lab 02: Task-to-Prompt Mapping
### Classification, Extraction, Summarization as engineering contracts

**Audience.** Practicing software engineers moving into AI engineering.

**Big idea.** A prompt is a code artifact. Its output has a contract, and a contract
is something you enforce with code you own. In this lab you pick the right task frame
for an input, then build the prompt, the output contract (a Pydantic model or a
structural validator), and the validation that turns a model response into a trusted
object or a clear error.

**What you will be able to do**
1. Map an input to the right task frame: classification, extraction, summarization, or generation.
2. Author a classification prompt with a label policy, tie-break, and abstain option, and enforce its output contract.
3. Design an extraction schema first, then validate strict JSON offline with Pydantic.
4. Write a summarization prompt with audience, scope, and style controls, and validate the result structurally.
5. Produce reusable output contracts and validators suitable for a continuous integration gate.

**Flow.** Part A frame selection, Part B classification, Part C extraction, Part D summarization, Part E round trip and wrap-up. Optional stretch goals follow.

> **Read before you run**
>
> - **The model is stubbed on purpose.** `call_model(...)` returns fixed, pre-captured
>   responses so this lab runs cold with no API keys and identical results every time.
>   The real-provider wiring is in the appendix at the end, marked read-only.
> - **The graded skill is the engineering around the prompt**, not a live call: prompt
>   construction, contracts, and validation. That is the part that is deterministic and testable.
> - **Checks are soft.** An unfinished exercise prints `ERROR` or `FAIL`, it never stops the
>   notebook. Work top to bottom. A full Restart-and-Run-All gives the true totals.
> - **Currency flag.** The appendix shows provider structured-output calls. Those APIs move.
>   Confirm the exact field and header names against current provider docs before teaching them live.

## Environment

Target stack: Python 3.13, pydantic 2.13.4, orjson 3.11.9. Run the next cell to
confirm your Pydantic version. Everything else in this lab is standard library.

In [ ]:
%pip install -r requirements.txt

In [ ]:
from __future__ import annotations

import re
import hashlib
from datetime import datetime

import orjson
import pydantic
from pydantic import BaseModel, ConfigDict, ValidationError, field_validator, model_validator

print("pydantic", pydantic.VERSION)

In [ ]:
# Soft check harness. Failing or unimplemented exercises are reported, never raised,
# so a Restart-and-Run-All stays clean. Totals accumulate across the notebook.
_RESULTS = {"pass": 0, "fail": 0, "error": 0}

def reset_checks():
    _RESULTS["pass"] = 0
    _RESULTS["fail"] = 0
    _RESULTS["error"] = 0

def check(label, fn):
    """fn is a zero-arg callable returning (ok: bool, detail: str)."""
    try:
        ok, detail = fn()
    except NotImplementedError:
        _RESULTS["error"] += 1
        print(f"[ ERROR ] {label}: not implemented yet")
        return
    except Exception as e:
        _RESULTS["error"] += 1
        print(f"[ ERROR ] {label}: {type(e).__name__}: {e}")
        return
    if ok:
        _RESULTS["pass"] += 1
        print(f"[ PASS  ] {label}")
    else:
        _RESULTS["fail"] += 1
        print(f"[ FAIL  ] {label}: {detail}")

def raises(fn):
    """True if fn() raises a validation or value error."""
    try:
        fn()
    except (ValidationError, ValueError):
        return True
    return False

def summary():
    r = _RESULTS
    print(f"\nTOTAL  pass={r['pass']}  fail={r['fail']}  error={r['error']}")

## Provided scaffolding

The next cells give you the fictional Cordwell Home and Hardware data, the label set,
the policy and contract text, the deterministic `call_model` stub, and the captured
responses used by the checks. You do not edit these. Read them so you know the shapes
you are targeting.

In [ ]:
ALLOWED_LABELS = ["billing", "bug", "performance", "account", "feature", "unknown"]

TICKETS = [
    {"ticket_id": "CW-101", "text": "Cordwell app crashes right after I tap 'Export order to PDF' on Android 14."},
    {"ticket_id": "CW-102", "text": "I was billed twice for my Pro Contractor plan this month even after I canceled last cycle."},
    {"ticket_id": "CW-103", "text": "Product search is very slow when my saved project has more than 5,000 line items."},
    {"ticket_id": "CW-104", "text": "Please add SSO with Okta so our whole procurement team can sign in without separate passwords."},
    {"ticket_id": "CW-105", "text": "Cannot reset my password. The reset link keeps saying the token is invalid."},
    {"ticket_id": "CW-106", "text": "My refund is still pending. The card was charged but the order never shipped."},
]

INVOICE_BLOCKS = [
    (
        "NORTHRIDGE BUILDING SUPPLY\n"
        "Invoice #: INV-7741\n"
        "Issue Date: 2025-07-03\n"
        "Due Date: 2025-07-17\n"
        "Subtotal: 4,875.00 USD\n"
        "Tax: 390.00 USD\n"
        "Total: 5,265.00 USD\n"
        "PO: PO-3319\n"
        "Vendor Email: billing@northridge-supply.example"
    ),
    (
        "BAYLIGHT SERVICES\n"
        "Invoice No: 00981\n"
        "Date: 03/22/2025\n"
        "Terms: Net 15\n"
        "Amount Due: USD 1,240.50\n"
        "Contact: ar@baylight.example"
    ),
]

REVIEWS = [
    {"review_id": "R-01", "text": (
        "The cordless drill has strong torque and the battery lasts through a full day of framing, "
        "but the charge indicator is off by about 10 percent. The grip is comfortable, though the "
        "chuck loosens under heavy load. Good for weekend jobs, not continuous professional use."
    )},
    {"review_id": "R-02", "text": (
        "This wet-dry shop vacuum has excellent suction and the filter is easy to clean; the hose is "
        "a little short at high reach. The motor housing heats up after 40 minutes of continuous use. "
        "Still, the price for the power you get is hard to beat."
    )},
]

FRAME_SCENARIOS = {
    "S1": "Route each inbound support ticket into exactly one queue from a fixed list.",
    "S2": "Pull the invoice number, total, and due date out of a scanned vendor invoice into fields.",
    "S3": "Turn ten long product reviews into a five-line briefing for a product manager.",
    "S4": "Given the label 'billing', the tie is between 'billing' and 'account'; decide which wins.",
    "S5": "Read a paragraph of release notes and produce a short marketing blurb from scratch.",
    "S6": "Decide whether a chat message is 'complaint', 'question', or 'praise'.",
}

SCHEMA_DESC = (
    "invoice_id (string, required) | issue_date (YYYY-MM-DD or null) | due_date (YYYY-MM-DD or null) | "
    "terms (string or null) | subtotal (number or null) | tax (number or null) | total (number, required) | "
    "po_number (string or null) | vendor_name (string, required) | vendor_email (string or null) | "
    "currency (string or null)"
)

CLASSIFICATION_POLICY = (
    "billing: charges, refunds, invoices. Counter: password errors are account.\n"
    "bug: broken or incorrect behavior. Counter: a request for something new is feature.\n"
    "performance: slowness or timeouts. Counter: a crash on a specific action is bug.\n"
    "account: login, profile, SSO, access. Counter: a card charge dispute is billing.\n"
    "feature: a request for new capability. Counter: a workaround for an existing feature.\n"
    "unknown: insufficient evidence.\n"
    "Tie-break: bug > performance > account > billing > feature.\n"
    "Abstain: if evidence is insufficient, choose 'unknown'."
)

SUMMARY_CONTRACT = (
    "### Cordwell PM Review Summary\n"
    "- Strengths: <comma-separated themes>\n"
    "- Weaknesses: <comma-separated themes>\n"
    "- Tradeoffs: <one line>\n"
    "- Priority fixes: <semicolon-separated, at most 3>\n"
    "- Quick wins: <semicolon-separated, at most 2>"
)

SUMMARY_HEADER = "### Cordwell PM Review Summary"
SUMMARY_MAX_WORDS_PER_BULLET = 14

# Provided utility: a simple email shape check you may use in Part C.
_EMAIL_RE = re.compile(r"[^@\s]+@[^@\s]+\.[^@\s]+")

In [ ]:
def call_model(task: str, prompt: str) -> str:
    """Deterministic stand-in for a hosted LLM call.

    Returns a fixed, pre-captured response per task so the lab runs cold with no
    keys and identical results every time. The real-provider version lives in the
    read-only appendix.
    """
    if task == "classification":
        return GOOD_CLASSIFICATION
    if task == "extraction":
        return GOOD_INVOICES
    if task == "summarization":
        return GOOD_SUMMARY
    raise ValueError(f"unknown task: {task!r}")

In [ ]:
GOOD_CLASSIFICATION = orjson.dumps({
    "records": [
        {"ticket_id": "CW-101", "label": "bug", "rationale": "The app crashes on tapping export to PDF, a broken behavior tied to a specific action."},
        {"ticket_id": "CW-102", "label": "billing", "rationale": "The customer reports being charged twice after canceling, which is a charges and refunds concern."},
        {"ticket_id": "CW-103", "label": "performance", "rationale": "Search is slow only on large saved projects, which points to a performance and timeout issue."},
        {"ticket_id": "CW-104", "label": "feature", "rationale": "The user asks to add Okta SSO, which is a request for a new capability rather than a defect."},
        {"ticket_id": "CW-105", "label": "account", "rationale": "Password reset fails with an invalid token, which is an access and account login problem."},
        {"ticket_id": "CW-106", "label": "billing", "rationale": "The card was charged but the order never shipped and a refund is pending, a billing dispute."},
    ],
    "stats": {"count": 6},
}).decode()

GOOD_INVOICES = orjson.dumps([
    {
        "invoice_id": "INV-7741", "issue_date": "2025-07-03", "due_date": "2025-07-17",
        "terms": None, "subtotal": 4875.00, "tax": 390.00, "total": 5265.00,
        "po_number": "PO-3319", "vendor_name": "Northridge Building Supply",
        "vendor_email": "billing@northridge-supply.example", "currency": "USD",
    },
    {
        "invoice_id": "00981", "issue_date": "2025-03-22", "due_date": None,
        "terms": "Net 15", "subtotal": None, "tax": None, "total": 1240.50,
        "po_number": None, "vendor_name": "Baylight Services",
        "vendor_email": "ar@baylight.example", "currency": "USD",
    },
]).decode()

GOOD_SUMMARY = (
    "### Cordwell PM Review Summary\n"
    "- Strengths: torque, battery life, suction power, easy filter cleaning\n"
    "- Weaknesses: charge indicator accuracy, chuck slippage, short hose, motor heat\n"
    "- Tradeoffs: strong value but durability and accuracy gaps limit continuous professional use\n"
    "- Priority fixes: charge indicator accuracy; chuck retention; motor thermal management\n"
    "- Quick wins: longer hose option; calibration note in specs"
)

In [ ]:
# Bad fixtures for classification (each should fail validation for one reason).
BAD_CLASSIFICATION = {
    "wrong_label": orjson.dumps({
        "records": [{"ticket_id": "CW-101", "label": "priority", "rationale": "x " * 12}],
        "stats": {"count": 1},
    }).decode(),
    "count_mismatch": orjson.dumps({
        "records": [{"ticket_id": "CW-101", "label": "bug", "rationale": "The app crashes on export which is a clear broken behavior worth triaging."}],
        "stats": {"count": 2},
    }).decode(),
    "rationale_too_short": orjson.dumps({
        "records": [{"ticket_id": "CW-101", "label": "bug", "rationale": "crashes on export"}],
        "stats": {"count": 1},
    }).decode(),
    "extra_key": orjson.dumps({
        "records": [{"ticket_id": "CW-101", "label": "bug", "rationale": "The app crashes on export which is a clear broken behavior worth triaging.", "confidence": 0.9}],
        "stats": {"count": 1},
    }).decode(),
}

# Bad fixtures for extraction.
BAD_INVOICES = {
    "missing_total": orjson.dumps([
        {"invoice_id": "INV-1", "vendor_name": "Acme"},
    ]).decode(),
    "bad_date": orjson.dumps([
        {"invoice_id": "INV-1", "issue_date": "July 3 2025", "total": 10.0, "vendor_name": "Acme"},
    ]).decode(),
    "bad_email": orjson.dumps([
        {"invoice_id": "INV-1", "total": 10.0, "vendor_name": "Acme", "vendor_email": "not-an-email"},
    ]).decode(),
    "extra_key": orjson.dumps([
        {"invoice_id": "INV-1", "total": 10.0, "vendor_name": "Acme", "discount": 5.0},
    ]).decode(),
    "total_wrong_type": orjson.dumps([
        {"invoice_id": "INV-1", "total": "ten dollars", "vendor_name": "Acme"},
    ]).decode(),
}

# Bad fixtures for summarization.
BAD_SUMMARY = {
    "missing_section": (
        "### Cordwell PM Review Summary\n"
        "- Strengths: torque, battery life\n"
        "- Weaknesses: charge indicator accuracy\n"
        "- Tradeoffs: strong value but accuracy gaps limit professional use\n"
        "- Priority fixes: charge indicator accuracy"
    ),
    "too_many_priority": (
        "### Cordwell PM Review Summary\n"
        "- Strengths: torque, battery life\n"
        "- Weaknesses: charge indicator accuracy\n"
        "- Tradeoffs: strong value but accuracy gaps limit professional use\n"
        "- Priority fixes: a; b; c; d\n"
        "- Quick wins: longer hose option"
    ),
    "bullet_too_long": (
        "### Cordwell PM Review Summary\n"
        "- Strengths: this strengths line intentionally runs far too long by listing many many many extra filler words here\n"
        "- Weaknesses: charge indicator accuracy\n"
        "- Tradeoffs: strong value but accuracy gaps limit professional use\n"
        "- Priority fixes: charge indicator accuracy\n"
        "- Quick wins: longer hose option"
    ),
    "has_quote": (
        "### Cordwell PM Review Summary\n"
        '- Strengths: torque, "excellent" battery life\n'
        "- Weaknesses: charge indicator accuracy\n"
        "- Tradeoffs: strong value but accuracy gaps limit professional use\n"
        "- Priority fixes: charge indicator accuracy\n"
        "- Quick wins: longer hose option"
    ),
}

## Part A - Frame selection

Before any prompt, decide the task frame. The wrong frame is the most expensive mistake
in the pipeline: it sends you down the wrong contract and the wrong validation.

Implement `pick_frame` so it returns the right frame for each scenario in
`FRAME_SCENARIOS`. The four frames are classification, extraction, summarization, and
generation. The check hides the answers, so reason it through rather than guessing.

🧑‍🏫 **Instructor note.** S4 is the trap. A tie between labels is still a classification
task, not extraction. Students who read "given the label" as data extraction will miss it.
S5 is the only generation item: output created from scratch, no fixed label set and nothing
to pull out. Use the hidden-digest check to explain why you would never ship an answer key
in a student notebook.

In [ ]:
def pick_frame(scenario_key: str) -> str:
    """Return the best task frame for FRAME_SCENARIOS[scenario_key].

    Output is exactly one of: "classification", "extraction", "summarization",
    "generation".
    """
    text = FRAME_SCENARIOS[scenario_key].lower()
    if "from scratch" in text or "blurb" in text:
        return "generation"
    if "into fields" in text:
        return "extraction"
    if "briefing" in text or "summar" in text:
        return "summarization"
    # Routing into a fixed set of buckets, or resolving a label tie, is classification.
    return "classification"

In [ ]:
_FRAME_DIGESTS = {
    "S1": "9f0c8aa526579f4dbde02f70f4cdf635a22df7d844ce012de5791febea95e732",
    "S2": "2d82451b16a8fed970ab776fd8483a8a3a91216b9275ed1772cc300a67c95cca",
    "S3": "7846ce4ebb5b98bf8b8065b8e15745dd2d3427d6f71a1864c269621e6232addf",
    "S4": "9f0c8aa526579f4dbde02f70f4cdf635a22df7d844ce012de5791febea95e732",
    "S5": "e661f4c935e8a5a83349afb5e347695c2e972e967b50efcd618f93b0b7b4c24b",
    "S6": "9f0c8aa526579f4dbde02f70f4cdf635a22df7d844ce012de5791febea95e732",
}

def check_frames():
    mismatched = []
    for k in FRAME_SCENARIOS:
        got = pick_frame(k)
        digest = hashlib.sha256(str(got).encode()).hexdigest()
        if digest != _FRAME_DIGESTS[k]:
            mismatched.append(k)
    return (not mismatched), f"wrong frame for: {mismatched}"

check("A. frame selection (all 6)", check_frames)

## Part B - Classification: policy, tie-break, abstain

You will triage the support tickets into exactly one label from `ALLOWED_LABELS`
(billing, bug, performance, account, feature, unknown).

Three pieces of work:
1. `build_classification_prompt` assembles the instruction, policy, contract, and tickets.
2. The Pydantic models encode the output contract: the allowed label set, the rationale
   length rule, unknown keys rejected, and the count that must match the number of records.
3. `validate_classification` turns a raw response string into a validated object or a raised error.

The contract you are enforcing:

```
{"records": [{"ticket_id": "<string>", "label": "<allowed>", "rationale": "<10 to 35 words>"}],
 "stats": {"count": "<int>"}}
```

🧑‍🏫 **Instructor note.** The rationale length rule is the teaching moment: a contract can
encode soft editorial rules, not just types. Show the `count_mismatch` fixture failing and
ask the room where that guard belongs. The tie-break line lives in the policy text so the
model has it, and Stretch 3 turns the same rule into deterministic post-processing you own.

In [ ]:
def build_classification_prompt(tickets: list[dict], policy: str) -> str:
    """Assemble the full classification prompt string.

    Must include: a triage role line, the label policy, the allowed label set, an
    instruction to return only JSON matching the contract (keys: records with
    ticket_id, label, rationale, and stats with count), and every ticket id and text.
    """
    lines = [
        "You are a careful support-ticket triage assistant for Cordwell Home and Hardware.",
        "",
        "Classify each ticket into exactly one label from this set:",
        ", ".join(ALLOWED_LABELS),
        "",
        "Label policy:",
        policy,
        "",
        "Return ONLY JSON matching this contract:",
        '{"records":[{"ticket_id":"<string>","label":"<allowed>","rationale":"<10-35 words>"}],"stats":{"count":"<int>"}}',
        "",
        "Tickets:",
    ]
    for t in tickets:
        lines.append(f'{t["ticket_id"]}: {t["text"]}')
    return "\n".join(lines)

In [ ]:
class ClassificationRecord(BaseModel):
    model_config = ConfigDict(extra="forbid")
    ticket_id: str
    label: str
    rationale: str

    @field_validator("label")
    @classmethod
    def _label_in_set(cls, v: str) -> str:
        if v not in ALLOWED_LABELS:
            raise ValueError(f"label {v!r} not in {ALLOWED_LABELS}")
        return v

    @field_validator("rationale")
    @classmethod
    def _rationale_length(cls, v: str) -> str:
        n = len(v.split())
        if not (10 <= n <= 35):
            raise ValueError(f"rationale must be 10-35 words, got {n}")
        return v


class ClassificationStats(BaseModel):
    model_config = ConfigDict(extra="forbid")
    count: int


class ClassificationResult(BaseModel):
    model_config = ConfigDict(extra="forbid")
    records: list[ClassificationRecord]
    stats: ClassificationStats

    @model_validator(mode="after")
    def _count_matches(self) -> "ClassificationResult":
        if self.stats.count != len(self.records):
            raise ValueError(f"stats.count {self.stats.count} != number of records {len(self.records)}")
        return self

In [ ]:
def validate_classification(raw: str) -> ClassificationResult:
    """Parse and validate a classification response. Raise on any contract violation."""
    return ClassificationResult.model_validate_json(raw)

In [ ]:
def check_class_prompt():
    p = build_classification_prompt(TICKETS, CLASSIFICATION_POLICY)
    need = [t["ticket_id"] for t in TICKETS] + ALLOWED_LABELS + ["JSON", "rationale", "Tie-break"]
    missing = [n for n in need if n not in p]
    return (not missing), f"prompt missing: {missing}"

def check_class_valid():
    res = validate_classification(GOOD_CLASSIFICATION)
    return (len(res.records) == 6 and res.stats.count == 6), "valid response should give 6 records"

def check_class_rejects():
    bad = {k: raises(lambda k=k: validate_classification(BAD_CLASSIFICATION[k])) for k in BAD_CLASSIFICATION}
    failed = [k for k, ok in bad.items() if not ok]
    return (not failed), f"did not reject: {failed}"

check("B1. classification prompt is complete", check_class_prompt)
check("B2/B3. validates a good response", check_class_valid)
check("B3. rejects every bad response", check_class_rejects)

## Part C - Extraction: schema first, strict JSON

Design the contract before the prompt. Implement `_normalize_date`, then the `Invoice`
model that encodes required fields, optional fields, date normalization, an email shape
check, and rejection of unknown keys. Then write `build_extraction_prompt` and
`validate_invoices`, which validates a JSON array one object at a time.

This replaces the hand-rolled regex validator from older versions of this lab with
Pydantic, which is in the cohort stack and gives you typed errors for free.

🧑‍🏫 **Instructor note.** `mode="before"` on the date validator is the point: normalization
runs before type coercion, so "03/22/2025" becomes "2025-03-22" and only then is stored.
Email uses a plain string with a regex validator because `EmailStr` needs the `email-validator`
package, which is not in the cohort stack. This is a good live example of verifying a
dependency before relying on it.

In [ ]:
def _normalize_date(value):
    if value is None:
        return None
    if not isinstance(value, str):
        raise ValueError("date must be a string or null")
    for fmt in ("%Y-%m-%d", "%m/%d/%Y"):
        try:
            return datetime.strptime(value, fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue
    raise ValueError(f"unparseable date: {value!r}")


class Invoice(BaseModel):
    model_config = ConfigDict(extra="forbid")
    invoice_id: str
    issue_date: str | None = None
    due_date: str | None = None
    terms: str | None = None
    subtotal: float | None = None
    tax: float | None = None
    total: float
    po_number: str | None = None
    vendor_name: str
    vendor_email: str | None = None
    currency: str | None = None

    @field_validator("issue_date", "due_date", mode="before")
    @classmethod
    def _norm_dates(cls, v):
        return _normalize_date(v)

    @field_validator("vendor_email")
    @classmethod
    def _valid_email(cls, v):
        if v is not None and not _EMAIL_RE.fullmatch(v):
            raise ValueError(f"invalid email: {v!r}")
        return v

In [ ]:
def build_extraction_prompt(blocks: list[str], schema_desc: str) -> str:
    """Assemble the extraction prompt.

    Must include: an extraction role, the schema description, rules (do not invent,
    use null when absent, normalize dates to YYYY-MM-DD), an instruction to return
    a strict JSON array with one object per block, and every invoice block delimited.
    """
    lines = [
        "You extract fields from invoice text into strict JSON.",
        "",
        "Schema (one object per invoice block):",
        schema_desc,
        "",
        "Rules:",
        "- Do not invent values. Use null when a field is absent.",
        "- Normalize every date to YYYY-MM-DD.",
        "- Return ONLY a JSON array, one object per block, no prose.",
        "",
        "Invoice blocks:",
    ]
    for i, block in enumerate(blocks, start=1):
        lines.append(f"[BLOCK {i}]")
        lines.append(block)
    return "\n".join(lines)

In [ ]:
def validate_invoices(raw: str) -> list[Invoice]:
    """Parse a JSON array and validate each object as an Invoice. Raise on any failure."""
    data = orjson.loads(raw)
    if not isinstance(data, list):
        raise ValueError("top-level JSON must be an array")
    return [Invoice.model_validate(item) for item in data]

In [ ]:
def check_extract_prompt():
    p = build_extraction_prompt(INVOICE_BLOCKS, SCHEMA_DESC)
    need = ["JSON", "null", "YYYY-MM-DD", "invoice_id", "vendor_name", "BLOCK 1", "BLOCK 2"]
    missing = [n for n in need if n not in p]
    return (not missing), f"prompt missing: {missing}"

def check_extract_valid():
    inv = validate_invoices(GOOD_INVOICES)
    ok = (len(inv) == 2 and inv[0].issue_date == "2025-07-03"
          and inv[1].issue_date == "2025-03-22" and inv[1].due_date is None
          and inv[0].total == 5265.00)
    return ok, f"good invoices should validate and normalize dates"

def check_extract_rejects():
    bad = {k: raises(lambda k=k: validate_invoices(BAD_INVOICES[k])) for k in BAD_INVOICES}
    failed = [k for k, ok in bad.items() if not ok]
    return (not failed), f"did not reject: {failed}"

check("C1/C3. validates good invoices and normalizes dates", check_extract_valid)
check("C2. extraction prompt is complete", check_extract_prompt)
check("C3. rejects every bad invoice set", check_extract_rejects)

## Part D - Summarization: audience, scope, style

Produce a briefing for Product Managers from the two reviews. The output is a fixed
markdown scaffold, so the validator is structural rather than a JSON schema.

Implement `build_summary_prompt` and `validate_summary`. The contract:

```
### Cordwell PM Review Summary
- Strengths: <comma-separated themes>
- Weaknesses: <comma-separated themes>
- Tradeoffs: <one line>
- Priority fixes: <semicolon-separated, at most 3>
- Quick wins: <semicolon-separated, at most 2>
```

Every bullet value is at most 14 words, and the summary contains no quote characters.

🧑‍🏫 **Instructor note.** Summaries are prose, so the contract is structural, not a JSON
schema. Walk the four bad fixtures: a missing section, too many priority fixes, an over-long
bullet, and a stray quote character. Each maps to one line of the validator. This is where
students see that "validator" is a general idea, not only Pydantic.

In [ ]:
def build_summary_prompt(reviews: list[dict], contract: str) -> str:
    """Assemble the summarization prompt.

    Must include: the target audience (Product Managers), the fixed markdown contract,
    the style and length constraints (at most 5 bullets, at most 14 words each, neutral
    tone, no quotes), and every review text.
    """
    lines = [
        "You summarize Cordwell product reviews for Product Managers planning a roadmap review.",
        "",
        "Constraints:",
        "- At most 5 bullets, at most 14 words each.",
        "- Neutral tone. No quotes. No customer names.",
        "",
        "Emit ONLY this contract:",
        contract,
        "",
        "Source reviews:",
    ]
    for r in reviews:
        lines.append(f'{r["review_id"]}: {r["text"]}')
    return "\n".join(lines)

In [ ]:
def validate_summary(md: str) -> dict:
    """Validate the summary against the markdown contract. Raise on any violation.

    Returns a dict with keys: strengths, weaknesses, tradeoffs, priority_fixes (list),
    quick_wins (list).
    """
    if '"' in md:
        raise ValueError("summary must not contain quote characters")
    lines = [ln.strip() for ln in md.strip().splitlines() if ln.strip()]
    if not lines or lines[0] != SUMMARY_HEADER:
        raise ValueError(f"first line must be {SUMMARY_HEADER!r}")

    fields = {}
    for ln in lines[1:]:
        if not ln.startswith("- "):
            raise ValueError(f"bullet lines must start with '- ': {ln!r}")
        body = ln[2:]
        if ": " not in body:
            raise ValueError(f"bullet must be 'Label: value': {ln!r}")
        label, value = body.split(": ", 1)
        if len(value.split()) > SUMMARY_MAX_WORDS_PER_BULLET:
            raise ValueError(f"bullet '{label}' exceeds {SUMMARY_MAX_WORDS_PER_BULLET} words")
        fields[label] = value

    required = ["Strengths", "Weaknesses", "Tradeoffs", "Priority fixes", "Quick wins"]
    missing = [r for r in required if r not in fields]
    if missing:
        raise ValueError(f"missing sections: {missing}")

    priority = [p.strip() for p in fields["Priority fixes"].split(";") if p.strip()]
    quick = [q.strip() for q in fields["Quick wins"].split(";") if q.strip()]
    if len(priority) > 3:
        raise ValueError(f"at most 3 priority fixes, got {len(priority)}")
    if len(quick) > 2:
        raise ValueError(f"at most 2 quick wins, got {len(quick)}")

    return {
        "strengths": fields["Strengths"],
        "weaknesses": fields["Weaknesses"],
        "tradeoffs": fields["Tradeoffs"],
        "priority_fixes": priority,
        "quick_wins": quick,
    }

In [ ]:
def check_summary_prompt():
    p = build_summary_prompt(REVIEWS, SUMMARY_CONTRACT)
    need = ["Product Managers", SUMMARY_HEADER, "14 words", "No quotes", "R-01", "R-02"]
    missing = [n for n in need if n not in p]
    return (not missing), f"prompt missing: {missing}"

def check_summary_valid():
    parsed = validate_summary(GOOD_SUMMARY)
    ok = (len(parsed["priority_fixes"]) == 3 and len(parsed["quick_wins"]) == 2)
    return ok, f"good summary should parse into sections"

def check_summary_rejects():
    bad = {k: raises(lambda k=k: validate_summary(BAD_SUMMARY[k])) for k in BAD_SUMMARY}
    failed = [k for k, ok in bad.items() if not ok]
    return (not failed), f"did not reject: {failed}"

check("D1. summary prompt is complete", check_summary_prompt)
check("D2. validates a good summary", check_summary_valid)
check("D2. rejects every bad summary", check_summary_rejects)

## Part E - Round trip and wrap-up

The payoff: build a prompt, send it through `call_model`, and validate the response into
a trusted object. In production the only change is swapping the stub for a real provider
call behind the same validation. Run the round trip, then print your totals.

In [ ]:
def check_roundtrip():
    prompt = build_classification_prompt(TICKETS, CLASSIFICATION_POLICY)
    raw = call_model("classification", prompt)
    res = validate_classification(raw)
    return (res.stats.count == 6), "round trip through call_model should validate"

check("E. end-to-end round trip (prompt to validated object)", check_roundtrip)
summary()

## Stretch goals (optional)

Each stretch goal extends the lab toward a production pattern. Implement any that interest you.

1. **Repair prompt.** `build_repair_prompt` consumes validator errors and asks the model to re-emit corrected JSON only.
2. **Severity linter.** `severity_rank` orders contract violations so the worst are addressed first.
3. **Tie-break resolver.** `apply_tie_break` turns the classification tie-break rule into deterministic code.
4. **Provider-aware structured output.** `structured_output_request` builds the request fragment that asks the provider to enforce the schema itself, which pairs with your own client-side validation.

🧑‍🏫 **Instructor note.** Stretch 4 is the currency anchor. The OpenAI and Anthropic request
shapes differ, and Anthropic's own shape changed from a beta header to a generally available
`output_config.format`. Use it to make the point that provider APIs move and your own
validation is the stable part.

In [ ]:
def build_repair_prompt(raw: str, errors: list[str]) -> str:
    """Stretch 1. Build a repair instruction that lists the errors and asks the model
    to re-emit corrected output as ONLY valid JSON, preserving all correct values."""
    bullet_errors = "\n".join(f"- {e}" for e in errors)
    return (
        "Your previous output failed validation. Fix exactly these problems:\n"
        f"{bullet_errors}\n\n"
        "Re-emit the corrected result as ONLY valid JSON matching the original contract. "
        "Preserve every value that was already correct. Do not add commentary.\n\n"
        "Previous output:\n"
        f"{raw}"
    )


_SEVERITY = {"structure": 0, "type": 1, "value": 2}


def severity_rank(errors: list[dict]) -> list[dict]:
    """Stretch 2. Sort contract errors by severity (structure first, then type, then
    value), breaking ties by field name. Each error is a dict with keys 'field',
    'kind' ('structure'|'type'|'value'), and 'msg'."""
    return sorted(errors, key=lambda e: (_SEVERITY.get(e["kind"], 99), e["field"]))


def apply_tie_break(candidates: list[str], priority: list[str]) -> str:
    """Stretch 3. Given several equally plausible labels, return the winner using the
    priority order (earlier wins). Labels not in priority lose to any that are; if none
    are in priority, return 'unknown'."""
    ranked = [c for c in priority if c in candidates]
    if ranked:
        return ranked[0]
    return "unknown"


def structured_output_request(schema: dict, provider: str) -> dict:
    """Stretch 4. Build a provider-appropriate structured-output request fragment.

    provider == 'openai': strict json_schema response_format.
    provider == 'anthropic': GA output_config.format with json_schema (no response_format).
    Raises for any other provider.
    """
    if provider == "openai":
        return {
            "response_format": {
                "type": "json_schema",
                "json_schema": {"name": "contract", "strict": True, "schema": schema},
            }
        }
    if provider == "anthropic":
        return {
            "output_config": {
                "format": {"type": "json_schema", "schema": schema},
            }
        }
    raise ValueError(f"unknown provider: {provider!r}")

In [ ]:
def check_repair():
    rp = build_repair_prompt('{"bad":1}', ["label 'priority' not allowed", "count mismatch"])
    ok = ("ONLY valid JSON" in rp and "priority" in rp and '{"bad":1}' in rp)
    return ok, "repair prompt must list errors and demand only valid JSON"

def check_severity():
    errs = [
        {"field": "total", "kind": "type", "msg": "x"},
        {"field": "records", "kind": "structure", "msg": "y"},
        {"field": "label", "kind": "value", "msg": "z"},
        {"field": "aaa", "kind": "structure", "msg": "w"},
    ]
    ranked = [e["field"] for e in severity_rank(errs)]
    return (ranked == ["aaa", "records", "total", "label"]), f"got order {ranked}"

def check_tiebreak():
    pr = ["bug", "performance", "account", "billing", "feature"]
    a = apply_tie_break(["billing", "account"], pr) == "account"
    b = apply_tie_break(["feature", "bug"], pr) == "bug"
    c = apply_tie_break(["nope"], pr) == "unknown"
    return (a and b and c), "tie-break must honor priority and fall back to unknown"

def check_structured():
    schema = {"type": "object"}
    o = structured_output_request(schema, "openai")
    a = structured_output_request(schema, "anthropic")
    ok = (o["response_format"]["json_schema"]["strict"] is True
          and "response_format" not in a
          and a["output_config"]["format"]["type"] == "json_schema")
    return ok, "provider request bodies must differ correctly"

check("S1. repair prompt", check_repair)
check("S2. severity rank", check_severity)
check("S3. tie-break resolver", check_tiebreak)
check("S4. provider-aware structured-output request", check_structured)

## Appendix - real provider wiring (read-only, do not run)

The lab is offline by design. When you move to a live call, the shape is below. Two
disciplines matter, and both survive whatever the provider APIs do next:

1. **Ask the provider to enforce the schema.** Modern providers can constrain output to a
   JSON schema. Your `structured_output_request` from the stretch goal builds this fragment.
2. **Validate on your side anyway.** Schema enforcement guarantees shape, not truth. Your
   Pydantic models and structural validators are the gate that decides what you trust.

> **Currency flag.** The exact field and header names below move. Confirm against current
> provider documentation before teaching this live.
>
> - **OpenAI** uses a strict `json_schema` `response_format`.
> - **Anthropic** now has a generally available path using `output_config.format` with a
>   `json_schema` and no beta header, alongside an older beta path that used the
>   `output_format` field with the `anthropic-beta: structured-outputs-2025-11-13` header.
> - **Do not** port OpenAI's `response_format` onto the Anthropic Messages API. It has never
>   existed there. Use the Anthropic field shown above.
> - Keep the model id in a config variable and confirm it is current. Do not hard-code a
>   model id you have not verified. Keep API keys in environment variables, never in code.

```python
# Illustrative only. Confirm field names, headers, and model id against current docs.
import os
# import anthropic  # provider SDK, not part of the offline cohort stack

MODEL = "confirm-current-model-id"  # e.g. a current Claude model; verify before use

def call_model_live(task: str, prompt: str) -> str:
    client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    schema = {...}  # the JSON schema for this task's contract
    req = structured_output_request(schema, "anthropic")  # from the stretch goal
    resp = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}],
        **req,
    )
    return resp.content[0].text
```

Even with a live call, you still run `validate_classification`, `validate_invoices`, or
`validate_summary` on the result. That is the responsible-AI seam: the validator is where
you check the output before anything downstream trusts it.

## Knowledge checks

1. Name one scenario where an abstain label prevents a forced misclassification.
2. Why does designing the schema before the prompt improve both output quality and your continuous integration gate?
3. Which elements of a summarization prompt control style and length in a way a validator can enforce?
4. Schema enforcement from a provider guarantees the shape of the output. What does it not guarantee, and which part of your code covers that gap?

## Self-assessment rubric (0 to 2 each, target 8 or higher)

- The chosen frame fits the input and the goal.
- The classification contract encodes label set, rationale length, no unknown keys, and the count rule.
- The extraction model validates good invoices, normalizes dates, and rejects every bad case.
- The summarization validator enforces the scaffold, the bullet counts, and the word budget.
- Prompts and validators are reproducible and free of leakage and vagueness.